In [1]:
import os
import time
import datetime
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loading dataset...")
dataset = load_dataset("ag_news", split="test")
samples = dataset.select(range(100))

# FIX 1: Correct label map — AG News label 3 is "Sci/Tech", NOT "Technology"
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# FIX 2: Prompt now exactly matches label names + stronger constraint
SYSTEM_PROMPT = (
    "You are a news classifier. "
    "Classify the text into EXACTLY one of these categories: World, Sports, Business, Sci/Tech. "
    "Reply with ONLY the category name. No punctuation, no explanation."
)

def classify_openai(text):
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o",  # FIX 3: thesis spec says GPT-4o, not gpt-4o-mini
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text: {text}"}
        ],
        max_tokens=10,
        temperature=0.0
    )
    latency = time.time() - start
    return response.choices[0].message.content.strip(), latency

def classify_anthropic(text):
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=10,
        temperature=0.0,  # FIX: match GPT-4o/Gemini's greedy decoding for a fair comparison
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Text: {text}"}]
    )
    latency = time.time() - start
    return response.content[0].text.strip(), latency

def classify_gemini(text):
    start = time.time()
    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",  # working model for this SDK version
            contents=f"{SYSTEM_PROMPT}\n\nText: {text}",
            config=types.GenerateContentConfig(
                max_output_tokens=10,
                temperature=0.0,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        latency = time.time() - start
        prediction = response.text.strip() if response.text else "Unknown"
        return prediction, latency
    except Exception as e:
        latency = time.time() - start
        print(f"Gemini error: {e}")
        return "Error", latency

# --- Evaluation ---
mlflow.set_experiment("ag_news_baseline")
results = []

with mlflow.start_run(run_name="baseline_v2_fixed_labels"):
    mlflow.log_param("sample_count", 100)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")
    mlflow.log_param("label_fix", "Sci/Tech_corrected")

    correct_oai = correct_ant = correct_gem = 0
    oai_lats = ant_lats = gem_lats = []
    oai_lats, ant_lats, gem_lats = [], [], []

    print("Evaluating...")
    for i, sample in enumerate(samples):
        text = sample["text"][:300]
        true_label = label_map[sample["label"]]

        oai_pred, oai_lat = classify_openai(text)
        ant_pred, ant_lat = classify_anthropic(text)
        gem_pred, gem_lat = classify_gemini(text)

        oai_lats.append(oai_lat)
        ant_lats.append(ant_lat)
        gem_lats.append(gem_lat)

        results.append({
            "text": text[:80],
            "true_label": true_label,
            "openai_pred": oai_pred,
            "anthropic_pred": ant_pred,
            "gemini_pred": gem_pred,
            "openai_correct": int(oai_pred == true_label),
            "anthropic_correct": int(ant_pred == true_label),
            "gemini_correct": int(gem_pred == true_label),
            "openai_latency_s": round(oai_lat, 3),
            "anthropic_latency_s": round(ant_lat, 3),
            "gemini_latency_s": round(gem_lat, 3),
        })

        if oai_pred == true_label: correct_oai += 1
        if ant_pred == true_label: correct_ant += 1
        if gem_pred == true_label: correct_gem += 1

        if i % 10 == 0:
            print(f"Progress: {i}/100")

    # Metrics
    oai_acc = correct_oai / 100
    ant_acc = correct_ant / 100
    gem_acc = correct_gem / 100

    mlflow.log_metric("openai_accuracy", oai_acc)
    mlflow.log_metric("anthropic_accuracy", ant_acc)
    mlflow.log_metric("gemini_accuracy", gem_acc)
    mlflow.log_metric("openai_avg_latency_s", sum(oai_lats) / len(oai_lats))
    mlflow.log_metric("anthropic_avg_latency_s", sum(ant_lats) / len(ant_lats))
    mlflow.log_metric("gemini_avg_latency_s", sum(gem_lats) / len(gem_lats))

    print(f"\n--- RESULTS ---")
    print(f"GPT-4o:            {oai_acc:.2%}")
    print(f"Claude Haiku:      {ant_acc:.2%}")
    print(f"Gemini 1.5 Flash:  {gem_acc:.2%}")

# Save results
os.makedirs("../logs", exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"../logs/ag_news_baseline_{timestamp}.csv"
pd.DataFrame(results).to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

Loading dataset...
Evaluating...
Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100

--- RESULTS ---
GPT-4o:            84.00%
Claude Haiku:      80.00%
Gemini 1.5 Flash:  85.00%
Saved to ../logs/ag_news_baseline_20260526_132945.csv
